# Module 1 · Embeddings — In-class Lab B 🧪
## Train a tiny word2vec

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mago-cinv/course-template/blob/master/modules/01-embeddings/labs/in-class/lab-b-word2vec.ipynb)

This is **Mini-lab B**, split out from the combined in-class lab. It pairs with **Lesson 1** (token embeddings).

### How this lab works

**The machinery is given to you.** The vocabulary builder, the skip-gram pair generator, the model, the negative-sampling loss and the whole training loop are written, commented and ready to run. You are *not* asked to re-derive `torch.bmm` shapes or look up Adam's signature — that is plumbing, and plumbing is not the lesson.

**What you write is the investigation.** Sections 🔬 **E3** and 🔬 **E4** hand you the same toolkit and ask you to *use* it: turn one knob at a time, measure what changes in the learned vectors, and defend a conclusion with numbers you produced.

Treat everything above the 🔬 sections as your **sandbox**: read it, run it, then take it apart.

Everything runs on CPU in a couple of minutes.


In [ ]:
# Setup — run me first (works on Colab AND locally)
# On Colab this installs the few extra libraries; locally the course venv
# (.venv, environment/requirements.txt) already has everything.
import sys

if "google.colab" in sys.modules:
    %pip install -q datasets tokenizers

print("Setup OK — running on", "Colab" if "google.colab" in sys.modules else "local Python")

In [ ]:
# Imports + seeds — fixed seeds make every run (and every student) identical
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

print("torch", torch.__version__)

## 1 · The corpus (provided)

Same AG News slice as Lab A — the first 2,000 lowercased texts.

In [ ]:
# Load the corpus: first 2,000 AG News training texts, lowercased
from datasets import load_dataset

ds = load_dataset("fancyzhx/ag_news", split="train").select(range(2000))
corpus = [t.lower() for t in ds["text"]]

print(f"{len(corpus)} texts")
print("sample:", corpus[0][:120], "…")
assert len(corpus) == 2000

## 2 · Your toolkit (provided)

The whole word2vec pipeline, packaged as six functions. **This is the sandbox** — E3 and E4 are built entirely from these.

| Tool | What it gives you |
|---|---|
| `build_vocab(V)` | word-level vocabulary + the corpus rewritten as id lists |
| `make_pairs(ids_corpus, C)` | all `(center, context)` training pairs for window half-size `C` |
| `untrained_loss(centers, contexts, k)` | the loss of a *fresh* model — the theory-check number |
| `train(centers, contexts, ...)` | build a fresh model and train it; returns `(model, epoch_losses)` |
| `embeddings(model)` | the matrix $E$ we keep (rows = center vectors $v_w$) |
| `nearest(word, E, k)` | the `k` most cosine-similar words — how we *judge* an embedding |

The knobs you will actually turn live in `make_pairs` (**`C`**, the window) and `train` (**`k`**, the negatives). Everything else is defaulted.

In [ ]:
# ══ THE TOOLKIT ══ everything below is provided; read it, then use it in E3 / E4


def build_vocab(V=2000, texts=None):
    """Keep the V most frequent whole words; rewrite the corpus as lists of ids.

    Classic word2vec used whole words (a modern system would feed Lab A's subwords).
    Rare words are simply dropped. Sets the globals used by `nearest`.
    """
    global vocab, word2id, id2word, counts
    texts = corpus if texts is None else texts
    counts = Counter(w for t in texts for w in t.split())
    vocab = [w for w, _ in counts.most_common(V)]
    word2id = {w: i for i, w in enumerate(vocab)}
    id2word = vocab                       # id -> word is just list indexing
    ids_corpus = [[word2id[w] for w in t.split() if w in word2id] for t in texts]
    return ids_corpus


def make_pairs(ids_corpus, C=2):
    """Slide a window of half-size C: every word is the center once, and each
    word within +-C of it becomes one (center, context) training pair.

        ...  my   old   PAPER   lantern   is  ...        C = 2
        pairs: (paper,my) (paper,old) (paper,lantern) (paper,is)

    C is the knob E3 investigates. Returns two aligned LongTensors.
    """
    centers, contexts = [], []
    for s in ids_corpus:
        for i, center in enumerate(s):
            for j in range(max(0, i - C), min(len(s), i + C + 1)):
                if j != i:
                    centers.append(center)
                    contexts.append(s[j])
    return torch.tensor(centers), torch.tensor(contexts)


class SkipGram(nn.Module):
    """Two embedding tables, no hidden nonlinearity (Lesson 1).

    in_embed  = E, the PROJECTION matrix: row w is the center vector v_w  <- the prize
    out_embed = U, the PREDICTION matrix: row w is the context vector u_w <- scaffolding
    Small init => every score u.v starts ~ 0 => the untrained loss lands on (1+k).ln2.
    """

    def __init__(self, V, d):
        super().__init__()
        self.in_embed = nn.Embedding(V, d)
        self.out_embed = nn.Embedding(V, d)
        nn.init.uniform_(self.in_embed.weight, -0.5 / d, 0.5 / d)
        nn.init.uniform_(self.out_embed.weight, -0.5 / d, 0.5 / d)


def make_sampler(power=0.75, k=5):
    """Draw 'fake' contexts from the unigram distribution raised to `power`.
    power=0.75 (Mikolov's choice) up-samples rare words relative to raw frequency."""
    freqs = torch.tensor([counts[w] for w in vocab], dtype=torch.float)
    p = freqs**power
    p = p / p.sum()
    return lambda n: torch.multinomial(p, n * k, replacement=True).view(n, k)


def neg_sampling_loss(model, center, context, sampler):
    """Lesson 1's eq. (5), averaged over the batch:

        L = -log sigma(u_o . v_w)  -  SUM_{w_n in N} log sigma(-u_{w_n} . v_w)

    first term  : push the score of the REAL context up
    second term : push the score of each FAKE context down
    F.logsigmoid is used instead of log(sigmoid(.)) — same math, numerically safe.
    """
    v_w = model.in_embed(center)                                 # (B, d)
    u_o = model.out_embed(context)                               # (B, d)
    pos = F.logsigmoid((u_o * v_w).sum(dim=1))                   # (B,)
    u_n = model.out_embed(sampler(len(center)))                  # (B, k, d)
    fake = torch.bmm(u_n, v_w.unsqueeze(2)).squeeze(2)           # (B, k)
    neg = F.logsigmoid(-fake).sum(dim=1)                         # (B,)
    return -(pos + neg).mean()


def untrained_loss(centers, contexts, k=5, power=0.75, d=50, seed=0, n=1024):
    """Loss of a FRESH (untrained) model. Theory says ~ (1+k)*ln 2 — E4 checks that."""
    torch.manual_seed(seed)
    model = SkipGram(len(vocab), d)
    with torch.no_grad():
        return neg_sampling_loss(model, centers[:n], contexts[:n], make_sampler(power, k)).item()


def train(centers, contexts, k=5, power=0.75, epochs=5, lr=5e-3,
          batch_size=1024, d=50, seed=0, verbose=True):
    """Build a fresh model and train it. Returns (model, epoch_losses).

    Re-seeds first, so the same arguments always give the same curve.
    k is the knob E4 investigates.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = SkipGram(len(vocab), d)
    sampler = make_sampler(power, k)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    for epoch in range(1, epochs + 1):
        perm = torch.randperm(len(centers))
        running = n_batches = 0
        for start in range(0, len(centers), batch_size):
            idx = perm[start : start + batch_size]
            loss = neg_sampling_loss(model, centers[idx], contexts[idx], sampler)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item()
            n_batches += 1
        losses.append(running / n_batches)
        if verbose:
            print(f"  epoch {epoch}: mean loss {losses[-1]:.3f}")
    return model, losses


def embeddings(model):
    """Keep E, discard U — the prediction head was scaffolding (Lesson 1)."""
    return model.in_embed.weight.detach()


def nearest(word, E, k=5):
    """The k most cosine-similar words to `word`. This is how we JUDGE an embedding."""
    En = E / E.norm(dim=1, keepdim=True).clamp_min(1e-8)   # unit rows -> dot = cosine
    sims = En @ En[word2id[word]]
    sims[word2id[word]] = -1.0                             # exclude the query itself
    top = sims.topk(k)
    return [(id2word[int(i)], round(float(s), 3)) for s, i in zip(top.values, top.indices)]


# Build the vocabulary once — every experiment below reuses it.
V = 2000
ids_corpus = build_vocab(V)
PROBES = ["company", "game", "president", "oil"]   # words we will judge the vectors on

print(f"toolkit ready — vocab {len(vocab)} words, "
      f"{sum(len(s) for s in ids_corpus):,} corpus tokens kept")
print(f"probe words for judging embeddings: {PROBES}")

## 3 · Worked example — from text to training pairs

The skip-gram game: *given a center word, predict its neighbours*. That's it. The "task" is a pretext — nobody cares about the predictions, we care about the vectors the model builds in order to make them.

*Expected:* on the order of $10^5$ pairs from 2,000 short texts, and decoded pairs that look like genuine neighbours.

In [ ]:
# Default window: C = 2
centers, contexts = make_pairs(ids_corpus, C=2)
print(f"{len(centers):,} (center, context) pairs at C=2\n")

for c, o in zip(centers[:6], contexts[:6]):
    print(f"  center = {id2word[int(c)]!r:12} context = {id2word[int(o)]!r}")

## 4 · Worked example — the untrained loss is a *prediction*

Before training, every score $u^\top v$ is ≈ 0 (that's what the small init buys us), so $\sigma(0)=\tfrac12$ and **each** of the $1+k$ terms in the loss contributes $\ln 2 \approx 0.693$.

$$\mathcal{L}_{\text{untrained}} \;\approx\; (1+k)\ln 2$$

With $k=5$ that predicts $6\ln 2 \approx 4.16$. This is Module 0's $\ln K$ sanity check, in this module's clothing — and E4 will test it properly.

*Expected:* measured and predicted agree to about two decimals.

In [ ]:
# Theory vs. measurement, at the default k = 5
k = 5
measured = untrained_loss(centers, contexts, k=k)
predicted = (1 + k) * np.log(2)

print(f"measured  untrained loss = {measured:.4f}")
print(f"predicted (1+k)·ln 2     = {predicted:.4f}")
print(f"difference               = {abs(measured - predicted):.4f}")

## 5 · Worked example — train it, and look at what it learned

Five epochs, Adam, mini-batches of 1,024 — seconds on CPU. Then we throw away $U$ and keep $E$, and judge the result the only way that matters: **do similar words end up near each other?**

*Expected:* the loss falls from ≈ 4.16 to ≈ 2 and flattens; the neighbours are roughly topical — rough (2,000 snippets, a few seconds of training!) but clearly not random.

In [ ]:
# Train once with the defaults
model, losses = train(centers, contexts, k=5, epochs=5)
E = embeddings(model)

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(losses) + 1), losses, marker="o", label="epoch-mean loss")
plt.axhline(measured, ls="--", c="gray", label=f"untrained ≈ {measured:.2f}")
plt.xlabel("epoch"); plt.ylabel("negative-sampling loss")
plt.xticks(range(1, len(losses) + 1))
plt.title("Tiny skip-gram: the loss falls, then flattens")
plt.legend(); plt.tight_layout(); plt.show()

print(f"E shape: {tuple(E.shape)}   (one {E.shape[1]}-d vector per vocabulary word)\n")
for q in PROBES:
    print(f"  nearest({q!r}) -> {[w for w, _ in nearest(q, E)]}")

---

## 🔬 Exercise E3 — What does the context window actually control?

We used `C=2` because we said so. The window decides **what "related" means**: a narrow window only ever pairs a word with its immediate grammatical neighbours, while a wide one pairs it with anything in the same sentence — that is, with its *topic*.

That's a claim. **Test it.**

**Design and run the experiment:**

1. Rebuild the pairs at several window sizes — e.g. `C ∈ [1, 2, 5, 10]` — with `make_pairs(ids_corpus, C=...)`.
2. For each: record **how many pairs** it produced, then `train(...)` (use `epochs=3` to keep the sweep quick) and take the `embeddings`.
3. For each window, print the `nearest(...)` list for every word in `PROBES`.
4. Store your results in a dict called `window_sweep` keyed by `C`.

**Then answer, in the markdown cell below:** how does the number of pairs scale with `C`? Looking at the neighbour lists, does a *wider* window give you more **topically** related words (things that appear in the same story) or more **grammatically interchangeable** ones (words that could swap places in a sentence)? Which `C` would you ship, and for what task?

> Everything you need is `make_pairs`, `train`, `embeddings` and `nearest`.

In [ ]:
# TODO: sweep the window size C and see how it changes the learned neighbours
# TODO: build `window_sweep` = {C: {"n_pairs": ..., "E": ..., "neighbors": {...}}, ...}
# TODO: print each window's nearest() lists for the words in PROBES
# HINT: centers_C, contexts_C = make_pairs(ids_corpus, C=c)
# HINT: model_C, _ = train(centers_C, contexts_C, epochs=3, verbose=False)
# HINT: nearest(q, embeddings(model_C)) for q in PROBES


In [ ]:
# Light check on YOUR experiment (in-class = completion, not correctness)
assert len(window_sweep) >= 3, "sweep at least 3 window sizes so a trend is visible"
assert all("n_pairs" in r and "neighbors" in r for r in window_sweep.values())

lo, hi = min(window_sweep), max(window_sweep)
assert window_sweep[hi]["n_pairs"] > window_sweep[lo]["n_pairs"], \
    "a wider window must produce MORE pairs"

ratio = window_sweep[hi]["n_pairs"] / window_sweep[lo]["n_pairs"]
print(f"✓ swept {len(window_sweep)} window sizes: C={lo} .. C={hi}")
print(f"  pairs grew {window_sweep[lo]['n_pairs']:,} -> {window_sweep[hi]['n_pairs']:,} ({ratio:.1f}x)")
overlap = [
    len(set(window_sweep[lo]["neighbors"][q]) & set(window_sweep[hi]["neighbors"][q]))
    for q in PROBES
]
print(f"  neighbour overlap between C={lo} and C={hi}: {sum(overlap)}/{5*len(PROBES)} words "
      f"— the window really did change what 'similar' means")

**Your reading of the sweep.** How did the pair count scale with `C`? Did wider windows give more topical or more grammatical neighbours? Which `C` would you ship?

> TODO: your answer here (3–4 sentences)


---

## 🔬 Exercise E4 — Put the theory on trial: does $(1+k)\ln 2$ hold?

Section 4 predicted the untrained loss from pure algebra and checked it at **one** value of $k$. One point is not a test of a formula — a broken formula can still be right once.

**Test it properly, then ask whether the knob is worth turning.**

**Part A — verify the prediction across the whole range.**

1. For `k ∈ [1, 2, 5, 10, 20]`, call `untrained_loss(centers, contexts, k=...)`.
2. Compare each against the predicted $(1+k)\ln 2$ — print both and the difference.
3. Plot measured vs. predicted (a straight line through the origin with slope $\ln 2$ is what "the theory is right" looks like).
4. Store the measurements in a dict called `k_sweep`.

**Part B — does more negatives mean better vectors?**

5. For at least three values of `k`, `train(centers, contexts, k=..., epochs=3)` and inspect `nearest(...)` on `PROBES`.
6. Report whether the neighbour quality visibly improves with `k` — and note that the *loss values are not comparable across `k`*, since a larger `k` simply sums more terms.

**Then answer below:** did the formula survive? And is `k` a knob worth tuning, or does it saturate?

> `untrained_loss`, `train`, `embeddings` and `nearest` are all you need.

In [ ]:
# TODO: Part A — check the untrained loss against (1+k)·ln 2 for several k
# TODO:   build `k_sweep` = {k: (measured, predicted), ...} and plot measured vs predicted
# TODO: Part B — train at 3+ values of k and compare nearest() neighbours on PROBES
# HINT: untrained_loss(centers, contexts, k=k)   and   (1 + k) * np.log(2)
# HINT: train(centers, contexts, k=k, epochs=3, verbose=False)


In [ ]:
# Light check on YOUR experiment (in-class = completion, not correctness)
assert len(k_sweep) >= 4, "test the formula at 4+ values of k"
assert all(len(v) == 2 for v in k_sweep.values()), "store (measured, predicted) per k"

errs = {k: abs(m - p) for k, (m, p) in k_sweep.items()}
worst = max(errs, key=errs.get)
print(f"✓ formula tested at k = {sorted(k_sweep)}")
print(f"  largest disagreement: {errs[worst]:.4f} at k={worst} "
      f"(measured {k_sweep[worst][0]:.3f} vs predicted {k_sweep[worst][1]:.3f})")
print(f"  mean absolute error : {np.mean(list(errs.values())):.4f}")
print("  → (1+k)·ln 2 " + ("HOLDS" if max(errs.values()) < 0.15 else "does NOT hold") +
      " across the tested range")

**Your verdict.** Did $(1+k)\ln 2$ survive the test? And is `k` worth tuning, or does its benefit saturate?

> TODO: your answer here (3–4 sentences)


---

## Wrap-up — what you actually did

You were handed a working word2vec sandbox and used it to produce two findings that were **not** in the notebook:

- **E3** — you showed that the context window is not a performance setting but a *definition*: it decides whether "similar" ends up meaning *grammatically interchangeable* or *appears in the same story*.
- **E4** — you put a closed-form prediction on trial across a whole range of $k$ instead of a single lucky point, and then checked whether the knob it describes is worth turning at all.

**Concept check (discuss aloud):**

1. Why is the untrained loss $(1+k)\ln 2$ and not just $\ln 2$?
2. Why can't you compare final loss values between a $k=1$ run and a $k=20$ run?
3. We throw away half the trained parameters ($U$) at the end. Why was it necessary to train them at all?

**Next:** Mini-lab C — now that you have vectors, what can you *do* with them?